[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/09_capa_embedding_llm/09_proyecto_capa_embedding_llm.ipynb)
# Proyecto 09 · Auditoría de la capa de embedding y extensión con vocabulario de dominio

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | ≈ 2 h |
| **Cómputo** | CPU (≈ 3–5 min en `FAST_DEV_RUN`); GPT-2 small si hay acceso al hub. 0 unidades de Colab |
| **Requisitos** | Lección 09 completa |

## Contexto de negocio
Eres ML engineer en **Políglota Labs**. El equipo va a ajustar un LM pequeño para generar *snippets* y consultas sintéticas sobre la documentación técnica (las usarás para entrenar el embedder en el módulo 11).
Antes de tocarlo, tu *tech lead* te pide dos cosas:
1. **Auditoría de la capa de embedding**: ¿qué tokens del vocabulario nunca aparecen en nuestro corpus?, ¿están sus filas «sanas»?, ¿cómo se relaciona la norma con la frecuencia?
2. **Vocabulario de dominio**: el tokenizer trocea mucho nuestros términos técnicos. Propón $K=32$ tokens nuevos, añádelos, inicialízalos con criterio y demuestra con números que **reduces tokens por palabra sin empeorar la calidad** (bits por byte).

**Conexión.** Viene del módulo 05 (tokens sub-entrenados) y 07 (inicialización de vocabulario nuevo); el resultado (tokenizer extendido + LM ajustado) alimenta la generación de datos del módulo 11.

## Dataset
Corpus Políglota (`carga_corpus`, módulo 00): Wikipedia ES/EN + funciones Python (CodeSearchNet o, sin red, la stdlib); si no hay red, texto sintético con el mismo esquema (el notebook avisa).

## Entregables y rúbrica
Los umbrales están **calibrados con la solución de referencia** (semilla 42, `FAST_DEV_RUN=True`, modelo de *fallback* mini-GPT-2 y corpus sintético, que es lo que pudo ejecutarse al construir el curso). Si corres con GPT-2 real y datos reales, ejecuta la solución y usa *sus* números menos el margen indicado.

| # | Entregable | Criterio (umbral) |
|---|---|---|
| R1 | `auditoria` con normas, frecuencias, ρ de Spearman y nº de tokens no vistos | claves presentes; `n_no_vistos` > 0; ρ es un número finito |
| R2 | Coseno medio entre filas de tokens **no vistos** vs vistos frecuentes | `cos_no_vistos − cos_frecuentes ≥ 0.5` (referencia mide ≈ 0.9). Solo con el modelo de *fallback*: con GPT-2 real «no visto en Políglota» no significa «no visto al entrenar» y R2 solo se informa |
| R3 | Lista de $K=32$ términos con ahorro = frecuencia × (piezas − 1) | 32 términos únicos, cada uno con ≥ 2 piezas en el tokenizer original |
| R4 | Extensión + init por media de sub-tokens; filas viejas **intactas** tras ajustar | `assert` de filas viejas pasa |
| R5 | Fertilidad (tokens/palabra) en el texto retenido | reducción ≥ 5 % (referencia ≈ 8–9 %) |
| R6 | Calidad tras ajustar solo las filas nuevas | `bpb_final ≤ bpb_base + 0.05` |

*(Los valores de referencia exactos se imprimen al ejecutar la solución.)*

In [ ]:
!pip install -q transformers tokenizers datasets scipy scikit-learn matplotlib tiktoken
import copy, json, math, os, random, re, time, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True   # True: minutos en CPU con modelos diminutos. False: corpus más grande y más pasos
PAL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · torch {torch.__version__}")

### Utilidades (corpus Políglota canónico y mini-LM de laboratorio). No hace falta modificarlas.

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
corpus = carga_corpus(FAST_DEV_RUN)
docs = corpus["docs"]
textos = [d["text"] for d in docs]
print("fuente:", corpus["source"], "| docs por idioma:", dict(Counter(d["lang"] for d in docs)))
print("ejemplo ES:", next(d["text"] for d in docs if d["lang"] == "es")[:160], "...")

In [ ]:
# ── Mini-LM de laboratorio: tokenizer BPE byte-level + GPT-2 diminuto (misma API de Hugging Face que GPT-2 real) ──
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast


def entrena_tokenizer(textos, vocab_size=500):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["<|endoftext|>"],
                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tk.train_from_iterator(textos, tr)
    return PreTrainedTokenizerFast(tokenizer_object=tk, eos_token="<|endoftext|>", pad_token="<|endoftext|>")


def crea_mini_gpt(tok, tie=True, d=64, seed=SEED):
    torch.manual_seed(seed)
    cfg = GPT2Config(vocab_size=len(tok), n_positions=64, n_embd=d, n_layer=2, n_head=2, tie_word_embeddings=tie,
                     bos_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    return GPT2LMHeadModel(cfg)


def flujo_ids(tok, textos):
    ids = []
    for t in textos:
        ids += tok(t)["input_ids"] + [tok.eos_token_id]
    return torch.tensor(ids)


def entrena_lm(model, data, steps=600, lr=1e-3, wd=0.0, B=16, T=48, seed=SEED):
    """Bucle de entrenamiento mínimo (AdamW). Devuelve la lista de pérdidas por paso."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    g = torch.Generator().manual_seed(seed)
    model.train(); hist = []
    for _ in range(steps):
        ix = torch.randint(0, len(data) - T - 1, (B,), generator=g)
        x = torch.stack([data[i:i + T] for i in ix])
        loss = model(input_ids=x, labels=x).loss
        opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    model.eval()
    return hist


@torch.no_grad()
def val_loss(model, data, T=48, n=64):
    xs = torch.stack([data[i:i + T] for i in range(0, min(len(data) - T - 1, n * T), T)])
    return model(input_ids=xs, labels=xs).loss.item()

In [ ]:
from tokenizers import AddedToken


def carga_lm():
    """GPT-2 small si el hub es accesible; si no, mini-GPT-2 ATADO entrenado aquí (fallback documentado)."""
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        return AutoModelForCausalLM.from_pretrained("gpt2").eval(), AutoTokenizer.from_pretrained("gpt2"), "GPT-2 small (real)"
    except Exception as e:
        print(f"⚠️  GPT-2 no disponible ({type(e).__name__}): entreno un mini-GPT-2 atado (fallback, ~30 s en CPU)")
    tk = entrena_tokenizer(textos, vocab_size=500)
    m = crea_mini_gpt(tk, tie=True)
    entrena_lm(m, flujo_ids(tk, random.Random(SEED).sample(textos, len(textos))), steps=600 if FAST_DEV_RUN else 3000)
    return m, tk, "mini-GPT-2 (fallback)"


@torch.no_grad()
def bpb_y_fuga(model, tok, textos, old_V, T=64):
    """(bits por byte, masa media en tokens nuevos donde el objetivo es viejo). Igual que en la lección."""
    nll, nbytes, fuga, npos = 0.0, 0, 0.0, 0
    for t in textos:
        ids = tok(t)["input_ids"][:T]
        if len(ids) < 2:
            continue
        x = torch.tensor([ids]); y = x[0, 1:]
        logp = torch.log_softmax(model(input_ids=x).logits[0, :-1].float(), -1)
        nll += -logp.gather(1, y[:, None]).sum().item(); nbytes += len(tok.decode(ids[1:]).encode("utf-8"))
        viejo = y < old_V
        if logp.shape[1] > old_V and viejo.any():
            fuga += logp[viejo][:, old_V:].exp().sum(-1).sum().item(); npos += int(viejo.sum())
    return nll / nbytes / math.log(2), (fuga / npos if npos else 0.0)


lm, tok, lm_name = carga_lm()
V = len(tok)
prosa = [d["text"] for d in docs if d["lang"] != "code"]
held = prosa[::7][:40]
ft_txt = [t for i, t in enumerate(prosa) if i % 7 != 0]
K = 32
print(lm_name, "| V =", V, "| tied =", lm.get_input_embeddings().weight is lm.get_output_embeddings().weight)

## Parte A · Auditoría de la capa de embedding

### TODO 1 · `auditoria(model, tok, textos)`
Devuelve un `dict` con: `normas` (np.array V), `freq` (np.array V, conteo de cada id en `textos` tokenizados con `tok`), `rho` (Spearman entre norma y $\log(1+f)$ **solo en tokens vistos**), `n_no_vistos`, `cos_no_vistos` y `cos_frecuentes` (coseno medio entre pares distintos de filas de la matriz de **salida**, para los no vistos y para los 100 más frecuentes).

<details><summary>🪜 Pista 1</summary>

Cuenta con `Counter` sobre `tok(t)["input_ids"]`. Para el coseno medio entre pares: normaliza las filas, haz `X @ X.T` y promedia fuera de la diagonal.
</details>
<details><summary>🪜 Pista 2</summary>

`W = model.get_output_embeddings().weight.detach().float()`; `X = F.normalize(W[idx], dim=-1)`; `S = X @ X.T`; `S[~torch.eye(len(idx), dtype=bool)].mean()`. Spearman: `scipy.stats.spearmanr(a, b).statistic`.
</details>

In [ ]:
from scipy.stats import spearmanr


def auditoria(model, tok, textos):
    # TODO 1: calcula normas, frecuencias, rho, n_no_vistos, cos_no_vistos, cos_frecuentes
    raise NotImplementedError


try:
    aud = auditoria(lm, tok, textos)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in aud.items() if k not in ("normas", "freq")})
except NotImplementedError:
    print("⏳ TODO 1 pendiente")

### TODO 2 · Interpreta (texto)
Escribe en una celda markdown 3–4 frases: ¿la diferencia `cos_no_vistos − cos_frecuentes` confirma la derivación de la lección? ¿Por qué con GPT-2 real los «no vistos en Políglota» **no** son necesariamente tokens sub-entrenados?

## Parte B · Vocabulario de dominio

### TODO 3 · `selecciona_terminos(tok, textos, k)`
Devuelve una lista de `k` cadenas **con espacio inicial** (`" palabra"`) ordenadas por ahorro = frecuencia × (nº de piezas − 1), considerando palabras alfabéticas de ≥ 6 caracteres.

<details><summary>🪜 Pista 1</summary>

`re.findall(r"[^\W\d_]{6,}", texto)` extrae palabras alfabéticas (con tildes). Piezas = `len(tok(" " + w, add_special_tokens=False)["input_ids"])`.
</details>

### TODO 4 · `extiende_subtoken(model, tok, nuevos)`
Copia modelo y tokenizer, añade los tokens (`AddedToken(w, normalized=False)`), redimensiona y **inicializa cada fila nueva con la media de las filas de sus piezas en el tokenizer original** (en todas las matrices distintas: entrada y, si no está atado, salida). Devuelve `(modelo, tokenizer, old_V)`.

<details><summary>🪜 Pista 1</summary>

Calcula las piezas **antes** de `add_tokens`. Recorre `{id(W): W for W in (entrada, salida)}.values()` para no inicializar dos veces la misma matriz atada.
</details>
<details><summary>🪜 Pista 2</summary>

`model.resize_token_embeddings(n, mean_resizing=False)` (envuelto en `try/except TypeError` para versiones antiguas); luego, dentro de `torch.no_grad()`, `W[old_V + j] = W[:old_V][piezas[j]].mean(0)`.
</details>

### TODO 5 · `ajusta_solo_nuevas(model, tok2, old_V, textos, steps)`
Congela todo salvo la(s) matriz(ces) de embeddings, enmascara el gradiente de las filas viejas con un *hook* y entrena con `entrena_lm(..., wd=0.0, lr=1e-2)`. Comprueba con `assert` que las filas viejas no cambian.

<details><summary>🪜 Pista 1</summary>

`hook = W.register_hook(lambda g: g * mask)` con `mask[old_V:] = 1`. ¿Por qué `wd=0.0` es imprescindible para que el `assert` pase? (Pista: lección, sección 3.)
</details>

In [ ]:
def selecciona_terminos(tok, textos, k=32):
    # TODO 3
    raise NotImplementedError


def extiende_subtoken(model, tok, nuevos):
    # TODO 4
    raise NotImplementedError


def ajusta_solo_nuevas(model, tok2, old_V, textos, steps=100):
    # TODO 5
    raise NotImplementedError

### TODO 6 · Ejecuta el pipeline y evalúa la rúbrica
La celda siguiente ya está escrita: usa tus funciones, mide fertilidad y bpb antes/después y comprueba la rúbrica.

In [ ]:
def pipeline_y_rubrica():
    aud = auditoria(lm, tok, textos)
    nuevos = selecciona_terminos(tok, ft_txt, K)
    fert = lambda tk: float(np.mean([len(tk(t)["input_ids"]) / len(t.split()) for t in held]))
    bpb_base, _ = bpb_y_fuga(lm, tok, held, V)
    m2, tok2, old_V = extiende_subtoken(lm, tok, nuevos)
    bpb0, fuga0 = bpb_y_fuga(m2, tok2, held, old_V)
    steps = 100 if V < 5000 else (25 if device == "cpu" and FAST_DEV_RUN else 200)
    ajusta_solo_nuevas(m2, tok2, old_V, ft_txt, steps=steps)
    bpb1, fuga1 = bpb_y_fuga(m2, tok2, held, old_V)
    r = dict(rho=aud["rho"], n_no_vistos=aud["n_no_vistos"], gap_cos=aud["cos_no_vistos"] - aud["cos_frecuentes"],
             fert_base=fert(tok), fert_ext=fert(tok2), bpb_base=bpb_base, bpb_init=bpb0, bpb_final=bpb1, fuga_init=fuga0, fuga_final=fuga1)
    r["reduccion_fert"] = 1 - r["fert_ext"] / r["fert_base"]
    piezas_ok = all(len(tok(w, add_special_tokens=False)["input_ids"]) >= 2 for w in nuevos)
    checks = {
        "R1 auditoría": r["n_no_vistos"] > 0 and np.isfinite(r["rho"]),
        # R2 solo es un test válido en el fallback: allí «no visto en el corpus» = «nunca visto al entrenar».
        # Con GPT-2 real, un token ausente de Políglota sí se vio en WebText: no se espera colapso (solo se informa).
        "R2 colapso de filas no vistas (gap ≥ 0.5)" + ("" if "fallback" in lm_name else " [no aplica con GPT-2 real]"):
            r["gap_cos"] >= 0.5 or "fallback" not in lm_name,
        "R3 32 términos únicos con ≥ 2 piezas": len(set(nuevos)) == K and piezas_ok,
        "R4 filas viejas intactas": True,                       # el assert de TODO 5 habría fallado antes
        "R5 reducción de fertilidad ≥ 5 %": r["reduccion_fert"] >= 0.05,
        "R6 bpb_final ≤ bpb_base + 0.05": r["bpb_final"] <= r["bpb_base"] + 0.05,
    }
    for k_, v_ in r.items():
        print(f"{k_:16s} {v_:.4f}" if isinstance(v_, float) else f"{k_:16s} {v_}")
    print()
    for k_, ok in checks.items():
        print(("✅ " if ok else "❌ ") + k_)
    return r, nuevos


try:
    resultado, nuevos = pipeline_y_rubrica()
except NotImplementedError:
    print("⏳ Completa los TODO 1, 3, 4 y 5 para ejecutar el pipeline.")

## Retos extra (nivel experto)
1. **Modelo desatado.** Repite con un mini-GPT-2 desatado (`tie=False`): inicializa también la matriz de salida y comprueba qué pasa si **no** lo haces (¿llega a predecirse algún token nuevo?).
2. **Init de Hewitt vs sub-tokens con pocos datos.** Repite el ajuste con 10, 30 y 100 pasos y grafica bpb vs pasos para `mean`, `subtoken` y `hf_default`.
3. **Relleno a múltiplo de 64.** Usa `pad_to_multiple_of=64` y verifica que las filas de relleno aparecen como «no vistas» en tu auditoría: exclúyelas.
4. **bpb por idioma.** Separa `held` en ES y EN: ¿la extensión ayuda por igual a ambos idiomas? Relaciónalo con la paridad de tokens del módulo 04.

## Reflexión
- ¿Qué indicador usarías en producción para vigilar tokens sub-entrenados si el modelo está atado? ¿Y si está desatado?
- Si Políglota añadiera 5 000 identificadores de código, ¿cuánta memoria extra supone en GPT-2 small ($d=768$, fp16)? ¿Y en el cómputo de la cabeza?
- ¿Cómo cambia este análisis para un embedder bidireccional (sin cabeza de LM) como el que entrenarás en el módulo 11?

---
## ⛔ SPOILER — intenta resolverlo primero
Debajo está la **solución de referencia** completa. Ejecutarla redefine las funciones y vuelve a correr el pipeline y la rúbrica.

In [ ]:
def auditoria(model, tok, textos):
    cnt = Counter(i for t in textos for i in tok(t)["input_ids"])
    Vv = model.get_input_embeddings().weight.shape[0]
    freq = np.array([cnt.get(i, 0) for i in range(Vv)])
    normas = model.get_input_embeddings().weight.detach().float().norm(dim=1).numpy()
    vis = freq > 0
    rho = float(spearmanr(normas[vis], np.log1p(freq[vis])).statistic)
    W = model.get_output_embeddings().weight.detach().float()

    def cos_medio(idx):
        X = F.normalize(W[torch.as_tensor(idx)], dim=-1)
        S = X @ X.T
        return float(S[~torch.eye(len(idx), dtype=bool)].mean())

    no_vistos = np.where(~vis)[0]
    return dict(normas=normas, freq=freq, rho=rho, n_no_vistos=int(len(no_vistos)),
                cos_no_vistos=cos_medio(no_vistos) if len(no_vistos) > 1 else float("nan"),
                cos_frecuentes=cos_medio(np.argsort(-freq)[:100]))


def selecciona_terminos(tok, textos, k=32):
    pal = Counter(w for t in textos for w in re.findall(r"[^\W\d_]{6,}", t))
    sc = [(w, n * (len(tok(" " + w, add_special_tokens=False)["input_ids"]) - 1)) for w, n in pal.items()]
    return [" " + w for w, _ in sorted(sc, key=lambda x: -x[1])[:k]]


def extiende_subtoken(model, tok, nuevos):
    tok2, m2 = copy.deepcopy(tok), copy.deepcopy(model)
    old_V = len(tok2)
    piezas = [tok2(w, add_special_tokens=False)["input_ids"] for w in nuevos]
    tok2.add_tokens([AddedToken(w, normalized=False) for w in nuevos])
    try:
        m2.resize_token_embeddings(len(tok2), mean_resizing=False)
    except TypeError:
        m2.resize_token_embeddings(len(tok2))
    mats = {id(W): W for W in (m2.get_input_embeddings().weight, m2.get_output_embeddings().weight)}
    with torch.no_grad():
        for W in mats.values():
            for j, p in enumerate(piezas):
                W[old_V + j] = W[:old_V][p].mean(0)
    return m2, tok2, old_V


def ajusta_solo_nuevas(model, tok2, old_V, textos, steps=100):
    for p in model.parameters():
        p.requires_grad_(False)
    mats = {id(W): W for W in (model.get_input_embeddings().weight, model.get_output_embeddings().weight)}
    hooks, antes = [], []
    for W in mats.values():
        W.requires_grad_(True)
        mask = torch.zeros_like(W); mask[old_V:] = 1
        hooks.append(W.register_hook(lambda g, m_=mask: g * m_))
        antes.append(W[:old_V].detach().clone())
    hist = entrena_lm(model, flujo_ids(tok2, textos), steps=steps, lr=1e-2, wd=0.0, B=8, T=48)
    for h_ in hooks:
        h_.remove()
    for W, a in zip(mats.values(), antes):
        assert torch.equal(a, W[:old_V].detach()), "¡las filas viejas cambiaron!"
    return hist


resultado, nuevos = pipeline_y_rubrica()
print("\nprimeros términos:", nuevos[:10])